# Coleta de Dados
## Ativo - ITUB4

In [1]:
# Importando as bibliotecas necessárias

import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

In [2]:
# Definindo o ticker/ativo e a data de início para a coleta dos dados
TICKER = 'ITUB4.SA'
# Definindo a data de início (5 anos atrás a partir de hoje)
START_DATE = (datetime.now() - timedelta(days=5*365)).strftime('%Y-%m-%d')
#Definindo a data de término (hoje)
END_DATE = datetime.now().strftime('%Y-%m-%d')


In [3]:
# Realizando o download dos dados históricos do ativo utilizando a biblioteca yfinance do ativo ITUB4.SA, com os ajustes automáticos para dividendos e splits, e especificando o período de 5 anos
df = yf.download(TICKER, auto_adjust=True, start=START_DATE, end=END_DATE)

[*********************100%***********************]  1 of 1 completed


In [4]:
# Removendo o droplevel do nível "Ticker"
df.columns = df.columns.droplevel("Ticker")

In [5]:
df.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2021-05-25,18.713394,18.958605,18.571431,18.919887,38124430
2021-05-26,18.971514,19.081213,18.765021,18.765021,23792660
2021-05-27,18.810184,19.036036,18.629504,18.971507,67821833
2021-05-28,19.119923,19.190903,18.713389,18.752107,29988923
2021-05-31,19.061846,19.313509,18.965052,19.145733,17563652


In [6]:
df.tail()

Price,Close,High,Low,Open,Volume
Date,,,,,
2026-05-18,39.619999,39.849998,39.299999,39.669998,22993100
2026-05-19,38.779999,39.419998,38.700001,39.049999,41150400
2026-05-20,39.669998,39.980000,39.099998,39.220001,38261500
2026-05-21,40.119999,40.450001,39.250000,39.430000,40949800
2026-05-22,39.430000,40.040001,39.310001,40.009998,21539700


In [7]:
df.shape

(1246, 5)

In [8]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1246 entries, 2021-05-25 to 2026-05-22
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   1246 non-null   float64
 1   High    1246 non-null   float64
 2   Low     1246 non-null   float64
 3   Open    1246 non-null   float64
 4   Volume  1246 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 58.4 KB


In [9]:
# Cálculo da Média Móvel Exponencial (EMA) de 60 dias
df['EMA_60'] = df['Close'].ewm(span=60, adjust=False).mean()

In [10]:
print(df[["Close", "EMA_60"]].head(65))
print(f"Valores nulos na EMA_60: {df['EMA_60'].isna().sum()}")

Price           Close     EMA_60
Date                            
2021-05-25  18.713394  18.713394
2021-05-26  18.971514  18.721857
2021-05-27  18.810184  18.724753
2021-05-28  19.119923  18.737709
2021-05-31  19.061846  18.748337
...               ...        ...
2021-08-19  19.388660  19.529949
2021-08-20  19.317318  19.522977
2021-08-23  19.524830  19.523038
2021-08-24  20.069523  19.540955
2021-08-25  20.095463  19.559136

[65 rows x 2 columns]
Valores nulos na EMA_60: 0


In [11]:
# Captando informações adicionais sobre o dataset, como valores ausentes, estatísticas descritivas e linhas com volume zerado
print("=== Valores ausentes ===")
print(df.isnull().sum())

print("\n=== Estatísticas descritivas ===")
print(df.describe())

print("\n=== Linhas com volume zerado ===")
zero_vol = df[df["Volume"] == 0]
print(f"Total: {len(zero_vol)}")
print(zero_vol)

=== Valores ausentes ===
Price
Close     0
High      0
Low       0
Open      0
Volume    0
EMA_60    0
dtype: int64

=== Estatísticas descritivas ===
Price        Close         High          Low         Open        Volume  \
count  1246.000000  1246.000000  1246.000000  1246.000000  1.246000e+03   
mean     24.228342    24.481763    23.977381    24.227386  3.283923e+07   
std       8.167008     8.233975     8.101689     8.170383  1.614542e+07   
min      13.783370    14.020222    13.757055    13.939855  6.695412e+06   
25%      17.843607    18.020588    17.643918    17.838143  2.196289e+07   
50%      21.820976    22.043298    21.643224    21.764094  2.944520e+07   
75%      27.832725    28.069048    27.524543    27.788616  4.030122e+07   
max      48.905388    49.202564    48.142634    48.588397  1.820390e+08   

Price       EMA_60  
count  1246.000000  
mean     23.667945  
std       7.527216  
min      15.166568  
25%      17.600094  
50%      19.944240  
75%      26.832762  
max   

In [12]:
df.head()

Price,Close,High,Low,Open,Volume,EMA_60
Date,,,,,,
2021-05-25,18.713394,18.958605,18.571431,18.919887,38124430,18.713394
2021-05-26,18.971514,19.081213,18.765021,18.765021,23792660,18.721857
2021-05-27,18.810184,19.036036,18.629504,18.971507,67821833,18.724753
2021-05-28,19.119923,19.190903,18.713389,18.752107,29988923,18.737709
2021-05-31,19.061846,19.313509,18.965052,19.145733,17563652,18.748337


In [13]:
# Removendo linhas com volume zerado
df = df[df["Volume"] > 0]
print(f"Dataset após remoção de linhas com volume zerado: {df.shape}")

Dataset após remoção de linhas com volume zerado: (1246, 6)


In [14]:
#Gerando arquivo CSV do dataset processado
df.to_csv("/Users/lucaszaninidasilva/Mackenzie/SetimoSemestre/IA/PROJETO/src/database/processed/ITUB4_processed.csv", index=True)